# Forecast Currency Exchange Rates for Risk Management

**Use Case:** EUR/USD exchange rate forecasting for risk management
**Dataset:** Real exchange rate data from benchmark cache (or synthetic fallback)
**API Plan:** Pro (uses auto model selection)

This notebook forecasts EUR/USD 30 days ahead and computes a risk probability band.


In [ ]:
import os, gzip, requests, random
import matplotlib.pyplot as plt

BASE_URL = "http://localhost:8000/v1"
HEADERS = {"Content-Type": "application/json", "X-Plan": "pro"}

def load_or_generate_exchange_rate():
    cache = os.path.join("..", "..", "benchmarks", ".cache", "exchange_rate.txt.gz")
    try:
        import pandas as pd
        with gzip.open(cache, "rt") as f:
            df = pd.read_csv(f, header=None, sep=",")
        series = df.iloc[:, 0].values[-200:].tolist()
        print("Loaded real Exchange Rate data.")
        return [round(float(v), 5) for v in series]
    except Exception:
        print("Generating synthetic EUR/USD series.")
        rng = random.Random(7)
        s = [1.0800]
        for _ in range(199):
            s.append(round(s[-1] * (1 + rng.gauss(0, 0.003)), 5))
        return s

series = load_or_generate_exchange_rate()
print(f"Series length: {len(series)} trading days")
print(f"Current rate: {series[-1]:.5f}")


## Step 1 — Forecast with Auto Model Selection

In [ ]:
response = requests.post(f"{BASE_URL}/forecast/univariate", headers=HEADERS,
    json={"series": series, "horizon": 30, "frequency": "D", "model": "auto",
          "confidence_levels": [0.8, 0.95]})
data = response.json()

print(f"Model selected : {data['model_used']}")
print(f"Fallback used  : {data['meta']['fallback_used']} ({data['meta']['fallback_reason']})")
print(f"Inference      : {data['meta']['inference_time_ms']}ms")
mean = data["forecast"]["mean"]
l95  = data["forecast"]["lower_95"]
u95  = data["forecast"]["upper_95"]
print(f"30-day forecast range: {min(mean):.5f} – {max(mean):.5f}")


## Step 2 — Visualize with 95% Confidence Band

In [ ]:
n = len(series)
fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor("white")

trim = max(0, n - 60)
hist_x = list(range(trim, n))
fore_x = list(range(n - 1, n + 30))
fore_mean = [series[-1]] + mean
fore_l95 = [series[-1]] + l95
fore_u95 = [series[-1]] + u95

ax.fill_between(fore_x, fore_l95, fore_u95, alpha=0.2, color="gray", label="95% CI")
ax.plot(hist_x, series[trim:], color="#2196F3", linewidth=1.5, label="Historical (last 60 days)")
ax.plot(fore_x, fore_mean, color="#FF9800", linewidth=2.5, linestyle="--",
        label=f"Forecast (30 days) — {data['model_used'].upper()}")

ax.set_title("EUR/USD Exchange Rate — 30-Day Forecast", fontsize=13, fontweight="bold")
ax.set_xlabel("Trading Day"); ax.set_ylabel("Exchange Rate")
ax.legend(loc="upper left"); ax.grid(True, alpha=0.3)
ax.set_facecolor("white")
plt.tight_layout()
plt.savefig("outputs/02_financial_forecast.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()


## Step 3 — Risk Management Insight

In [ ]:
lo = round(min(l95), 5)
hi = round(max(u95), 5)
current = series[-1]
worst_loss = round((current - lo) / current * 100, 2)
best_gain  = round((hi - current) / current * 100, 2)

print("=" * 50)
print("Risk Management Summary (next 30 trading days)")
print("=" * 50)
print(f"Current rate     : {current:.5f}")
print(f"Forecast range   : {min(mean):.5f} – {max(mean):.5f}")
print()
print(f"95% probability: rate stays between {lo:.5f} and {hi:.5f}")
print(f"  Downside risk  : -{worst_loss}% vs. current")
print(f"  Upside chance  : +{best_gain}% vs. current")
print()
print("Recommendation: hedge positions with max {:.0f}% downside buffer.".format(worst_loss * 1.2))
